# Lab 6: Training and Analyzing a Transformer Classifier

**Course:** CS 417 — Large Language Models

**Name:** Muhammad Haseeb Ul Haq &nbsp;&nbsp;&nbsp; **Registration No.:** 454512

**Prerequisites:** This lab builds directly on Lab 5. The AG News subsets, the BPE tokenizer, `ignore_padding`, `positional_encoding` and the `TransformerEncoderBlock` are given below exactly as built there, so you do not need to re-implement them. If you prefer, replace the given encoder class with your own Lab 5 class, as long as it still passes the check under it.

In Lab 5 the encoder was never trained. In this lab you turn it into a complete **four-class news classifier**, train it, and look at what changed. You will:

- average only the real-token representations into one document vector;
- add a classification layer and assemble the full model;
- train for two epochs and evaluate on held-out test data;
- compare attention before and after training, and compare a token's embedding with its contextualized representation;
- in an extension, fine-tune a **pretrained** Hugging Face encoder on the same task and compare the two approaches.

The goal is to show that your hand-built encoder **can learn** and to understand what it learned, not to maximize accuracy.

**Implementation rules (Tasks 1–7):** do not use `nn.MultiheadAttention`, `nn.Transformer`, `nn.TransformerEncoder`, `nn.TransformerEncoderLayer` or `torch.nn.functional.scaled_dot_product_attention`. Ready-made library models are allowed **only in Task 8**.

In [ ]:
# Required packages: torch, matplotlib, datasets, tokenizers
# (Task 8 also needs transformers and accelerate.)
# On Google Colab, uncomment the next line if anything is missing.
# !pip install -q datasets tokenizers transformers accelerate

import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

# ---------------- Configuration (use these values) ----------------
SEED          = 42
MAX_LENGTH    = 32
VOCAB_SIZE    = 8000
BATCH_SIZE    = 16
D_MODEL       = 64
NUM_HEADS     = 4
D_HEAD        = D_MODEL // NUM_HEADS   # = 16
D_FF          = 256
NUM_CLASSES   = 4
NUM_EPOCHS    = 2
LEARNING_RATE = 1e-3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("device:", device)

---
## Setup — Restore the Lab 5 Pipeline (given)

The next three cells rebuild everything from Lab 5: the same disjoint subsets (same seed, same procedure), the same BPE tokenizer trained on the **training text only**, fixed-length token IDs, and the encoder. Run them; there is nothing to implement here.

If you saved your Lab 5 tokenizer as `lab05_bpe_tokenizer.json`, upload it to this session and it will be loaded instead of retrained.

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

raw = load_dataset("fancyzhx/ag_news")   # also available as "ag_news"
LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
N_TRAIN, N_VAL, N_TEST = 4000, 1000, 1000

# Same subset procedure as Lab 5: disjoint train/validation from the train split, test from the test split
rng = random.Random(SEED)
train_all = list(range(len(raw["train"])))
rng.shuffle(train_all)
train_idx = train_all[:N_TRAIN]
val_idx = train_all[N_TRAIN:N_TRAIN + N_VAL]
test_all = list(range(len(raw["test"])))
rng.shuffle(test_all)
test_idx = test_all[:N_TEST]

train_raw = raw["train"].select(train_idx)
val_raw = raw["train"].select(val_idx)
test_raw = raw["test"].select(test_idx)

train_texts, train_labels = list(train_raw["text"]), list(train_raw["label"])
val_texts, val_labels = list(val_raw["text"]), list(val_raw["label"])
test_texts, test_labels = list(test_raw["text"]), list(test_raw["label"])

# Same BPE tokenizer as Lab 5 (trained on training text only)
TOKENIZER_FILE = "lab05_bpe_tokenizer.json"
if os.path.exists(TOKENIZER_FILE):
    tokenizer = Tokenizer.from_file(TOKENIZER_FILE)
    print("loaded tokenizer from", TOKENIZER_FILE)
else:
    tokenizer = Tokenizer(models.BPE(unk_token="<UNK>", continuing_subword_prefix="##"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<PAD>", "<UNK>"],
                                  continuing_subword_prefix="##")
    tokenizer.train_from_iterator(train_texts, trainer)
    tokenizer.save(TOKENIZER_FILE)
    print("trained tokenizer and saved it to", TOKENIZER_FILE)

pad_id = tokenizer.token_to_id("<PAD>")
unk_id = tokenizer.token_to_id("<UNK>")
tokenizer.enable_truncation(max_length=MAX_LENGTH)
tokenizer.enable_padding(pad_id=pad_id, pad_token="<PAD>", length=MAX_LENGTH)


def encode_texts(texts):
    return torch.tensor([e.ids for e in tokenizer.encode_batch(texts)], dtype=torch.long)


train_ids, val_ids, test_ids = encode_texts(train_texts), encode_texts(val_texts), encode_texts(test_texts)
train_valid, val_valid, test_valid = train_ids != pad_id, val_ids != pad_id, test_ids != pad_id

# The same text must always give the same token IDs
assert torch.equal(encode_texts(train_texts[:3]), train_ids[:3])
print("vocabulary size:", tokenizer.get_vocab_size(), "| <PAD> id:", pad_id, "| <UNK> id:", unk_id)
print("train / val / test ids:", tuple(train_ids.shape), tuple(val_ids.shape), tuple(test_ids.shape))

In [ ]:
# ---------------- Given: Lab 5 building blocks ----------------

def ignore_padding(scores, valid_tokens):
    # Instructor-provided helper (unchanged from Lab 5)
    if scores.ndim == 3:
        valid_keys = valid_tokens[:, None, :]
    elif scores.ndim == 4:
        valid_keys = valid_tokens[:, None, None, :]
    else:
        raise ValueError("Expected 3D or 4D attention scores")
    return scores.masked_fill(~valid_keys, float("-inf"))


def positional_encoding(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float()
    i2 = torch.arange(0, d_model, 2).float()
    div = torch.pow(10000.0, i2 / d_model)
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(pos / div)
    pe[:, 1::2] = torch.cos(pos / div)
    return pe


class TransformerEncoderBlock(nn.Module):
    # Same as Lab 5, Task 7. You may replace this class with your own Lab 5 version.
    def __init__(self, d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF, dropout=0.1):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, L, D = x.shape
        return x.view(B, L, self.num_heads, self.d_head).transpose(1, 2)

    def combine_heads(self, x):
        B, H, L, dh = x.shape
        return x.transpose(1, 2).reshape(B, L, H * dh)

    def forward(self, x, valid_tokens, return_attention=False):
        Q = self.split_heads(self.W_Q(x))
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        scores = ignore_padding(scores, valid_tokens)
        attn = torch.softmax(scores, dim=-1)
        context = self.dropout(attn) @ V
        attn_out = self.W_O(self.combine_heads(context))
        x1 = self.norm1(x + attn_out)
        out = self.norm2(x1 + self.ffn(x1))
        return (out, attn) if return_attention else out


# Quick check of the encoder (also run this if you pasted your own class)
_enc = TransformerEncoderBlock().eval()
_x = torch.randn(4, MAX_LENGTH, D_MODEL)
with torch.no_grad():
    _out, _attn = _enc(_x, train_valid[:4], return_attention=True)
assert _out.shape == (4, MAX_LENGTH, D_MODEL) and _attn.shape == (4, NUM_HEADS, MAX_LENGTH, MAX_LENGTH)
assert torch.allclose(_attn.sum(-1), torch.ones(4, NUM_HEADS, MAX_LENGTH), atol=1e-5)
print("Lab 5 encoder check passed.")

In [ ]:
# ---------------- Given: data loaders ----------------
train_loader = DataLoader(TensorDataset(train_ids, torch.tensor(train_labels)), batch_size=BATCH_SIZE,
                          shuffle=True, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(TensorDataset(val_ids, torch.tensor(val_labels)), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(test_ids, torch.tensor(test_labels)), batch_size=BATCH_SIZE)

ids_batch, labels_batch = next(iter(train_loader))
print("one batch:", tuple(ids_batch.shape), tuple(labels_batch.shape))
print("batches per epoch:", len(train_loader))

---
## Task 1 — From Token Representations to a Document Representation

The encoder returns one vector **per token**: `[B, 32, 64]`. To classify an article we need one vector **per article**: `[B, 64]`.

Use **mean pooling over the real (non-padding) tokens only**:

$$\text{doc} = \frac{1}{n}\sum_{i=1}^{n} h_i, \qquad n = \text{number of real tokens}$$

For `[AI, changes, business, <PAD>, <PAD>]`, add the first three encoder vectors and divide by **3**, not 5.

**TODO:** Implement `masked_mean_pool(encoder_output, valid_tokens)`:

1. Expand `valid_tokens` from `[B, L]` to `[B, L, 1]`.
2. Convert it to the encoder output's number type (`.to(encoder_output.dtype)`), so True/False become 1.0/0.0.
3. Multiply the encoder output by it, which zeroes every padding vector.
4. Sum over the token dimension and divide by the number of real tokens (use `.clamp(min=1)` on the count so nothing ever divides by zero).

In [ ]:
def masked_mean_pool(encoder_output, valid_tokens):
    mask = valid_tokens.unsqueeze(-1).to(encoder_output.dtype)   # [B, L, 1], 1.0/0.0
    summed = (encoder_output * mask).sum(dim=1)                   # zero out padding, then sum
    real_counts = mask.sum(dim=1).clamp(min=1)                    # [B, 1] real tokens per article
    return summed / real_counts


# --- test your implementation ---
h = torch.arange(10, dtype=torch.float).reshape(1, 5, 2)        # 1 article, 5 tokens, 2 dims
mask = torch.tensor([[True, True, True, False, False]])          # 3 real tokens, 2 padding
pooled = masked_mean_pool(h, mask)
print("token vectors:\n", h[0])
print("pooled:", pooled)
assert pooled.shape == (1, 2)
assert torch.allclose(pooled[0], h[0, :3].mean(dim=0)), "divide by the number of REAL tokens (3), not 5"

h_noisy = h.clone()
h_noisy[0, 3:] = 1000.0                                          # change only the padding vectors
assert torch.allclose(masked_mean_pool(h_noisy, mask), pooled), "padding vectors must not affect the result"

torch.manual_seed(SEED)
fake_out = torch.randn(BATCH_SIZE, MAX_LENGTH, D_MODEL)
doc = masked_mean_pool(fake_out, train_valid[:BATCH_SIZE])
assert doc.shape == (BATCH_SIZE, D_MODEL)
print("batch:", tuple(fake_out.shape), "->", tuple(doc.shape))
print("max difference from a naive mean over all 32 positions:",
      (doc - fake_out.mean(dim=1)).abs().max().item())


#### What I Did

I expand `valid_tokens` to `[B, L, 1]` and cast it to float so real positions become 1.0 and padding becomes 0.0, multiply it into the encoder output so every padding vector becomes exactly zero, sum over the token dimension, and divide by the real-token count (clamped to at least 1 so an article can never divide by zero). The noisy-padding test confirms padding truly has no effect: changing the padding vectors to 1000 doesn't move the pooled result at all.

**Analysis Question**

- **Q1.** Why must the denominator be the number of real tokens rather than 32? The `<PAD>` embedding is all zeros, so why is it still necessary to exclude padding positions **after** the encoder? *Hint: what happens to a padding position after positional encoding is added and after the encoder's residual, LayerNorm and FFN steps?*

#### My Answers

- **Q1.** Dividing by 32 would silently dilute every document vector by however much padding that particular article happens to have - a short, 10-token article would have its real content's contribution shrunk by more than two-thirds just because of how much `<PAD>` padding it needed, which has nothing to do with the article's actual content. Dividing by the real-token count keeps the average comparable across articles of different lengths. Excluding padding *after* the encoder still matters even though the `<PAD>` embedding itself starts at exactly zero, because a padding position doesn't stay zero: positional encoding adds a nonzero position vector to it first, and then LayerNorm (which recenters/rescales every position to mean 0, variance 1) and the FFN's bias terms both turn that zero-plus-position vector into something nonzero by the time it leaves the encoder. By the time pooling happens, padding positions hold real, nonzero numbers that have nothing to do with the article's content, so they must be masked out explicitly rather than relied on to already be zero.

---
## Task 2 — The Classification Layer and the Complete Model

A linear layer `nn.Linear(64, 4)` maps each document vector to **four raw scores (logits)**, one per AG News category. The largest logit is the predicted class.

```text
Token IDs [B, 32]
 → token embedding + positional encoding   [B, 32, 64]
 → TransformerEncoderBlock                 [B, 32, 64]
 → mean pooling over real tokens           [B, 64]
 → linear classifier                       [B, 4]
```

**TODO:** Complete `NewsClassifier`.

- `__init__`: an embedding layer (`padding_idx=pad_id`), the positional-encoding matrix stored with `self.register_buffer("pe", ...)` so it moves to the GPU with the model, one `TransformerEncoderBlock`, and `nn.Linear(D_MODEL, NUM_CLASSES)` as `self.classifier`.
- `forward(token_ids, return_details=False)`: build `valid_tokens = token_ids != self.pad_id` **inside** the model, run the pipeline above, and return the **raw logits** (no softmax). When `return_details=True`, also return a dictionary with the keys `"token_embeddings"`, `"encoder_input"`, `"encoder_output"`, `"attention"`, `"valid_tokens"` and `"document"`. Later tasks use them for analysis.

In [ ]:
class NewsClassifier(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF,
                 num_classes=NUM_CLASSES, max_length=MAX_LENGTH, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.register_buffer("pe", positional_encoding(max_length, d_model))
        self.encoder = TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, token_ids, return_details=False):
        valid_tokens = token_ids != self.pad_id

        token_embeddings = self.embedding(token_ids)
        L = token_ids.shape[1]
        encoder_input = token_embeddings + self.pe[:L]

        encoder_output, attention = self.encoder(encoder_input, valid_tokens, return_attention=True)
        document = masked_mean_pool(encoder_output, valid_tokens)
        logits = self.classifier(document)

        if return_details:
            details = {
                "token_embeddings": token_embeddings,
                "encoder_input": encoder_input,
                "encoder_output": encoder_output,
                "attention": attention,
                "valid_tokens": valid_tokens,
                "document": document,
            }
            return logits, details
        return logits


# --- test your implementation ---
torch.manual_seed(SEED)
model = NewsClassifier(tokenizer.get_vocab_size(), pad_id).to(device)

token_ids = train_ids[:BATCH_SIZE].to(device)
model.eval()
with torch.no_grad():
    logits, details = model(token_ids, return_details=True)
    plain = model(token_ids)

print("Token IDs:     ", tuple(token_ids.shape))
for k in ["token_embeddings", "encoder_input", "encoder_output", "attention", "valid_tokens", "document"]:
    print(f"{k:17s}", tuple(details[k].shape))
print("Logits:        ", tuple(logits.shape))

B = token_ids.shape[0]
assert torch.is_tensor(plain) and plain.shape == (B, NUM_CLASSES), "without return_details, return only the logits"
assert logits.shape == (B, NUM_CLASSES) and torch.isfinite(logits).all()
assert details["document"].shape == (B, D_MODEL)
assert details["attention"].shape == (B, NUM_HEADS, MAX_LENGTH, MAX_LENGTH)
vt = details["valid_tokens"]
assert (details["attention"] * (~vt)[:, None, None, :]).abs().max().item() < 1e-6, \
    "padding keys must receive ~0 attention"
assert isinstance(model.classifier, nn.Linear) and model.classifier.out_features == NUM_CLASSES
assert "pe" in dict(model.named_buffers()), "store the positional encoding with register_buffer"
banned = (nn.MultiheadAttention, nn.Transformer, nn.TransformerEncoder, nn.TransformerEncoderLayer)
assert not any(isinstance(m, banned) for m in model.modules())
custom_params = sum(p.numel() for p in model.parameters())
print("\nAll model checks passed. Trainable parameters:", custom_params)


#### What I Did

`NewsClassifier` wires together everything: an embedding layer with `padding_idx=pad_id`, the Lab 5 positional encoding stored as a buffer (so it moves with the model to the GPU but is never trained), one `TransformerEncoderBlock`, and a `Linear(64, 4)` classifier. `forward` computes `valid_tokens` itself from the raw token IDs, runs the full pipeline down to masked mean pooling, and returns raw logits - with `return_details=True` it also hands back every intermediate tensor later tasks need for analysis.

**Analysis Question**

- **Q2.** Why does the classifier produce four output values for every document? What does the largest of the four mean, and why are they called *logits* rather than probabilities?

#### My Answers

- **Q2.** There are four output values because AG News has four categories, and `nn.Linear(64, 4)` is defined to produce exactly one score per category. The largest of the four logits is the model's prediction - whichever class got the highest raw score is the class it's most confident about. They're called logits rather than probabilities because they're unconstrained real numbers (they can be negative, and they don't sum to anything in particular); softmax is the specific function that would turn them into a proper probability distribution (all non-negative, summing to 1), and that conversion is deliberately left for the loss function to do internally rather than being applied here.

---
## Task 3 — Train the Model

### 3.1 Save the attention pattern *before* training

To compare attention before and after training (Task 5), first record what the **untrained** model does on one fixed **validation** article. The test subset is kept for final evaluation only.

**TODO:** Choose `VAL_EX` (a validation article index) and `HEAD` (0–3). Put the model in evaluation mode and, inside `torch.no_grad()`, run that one article with `return_details=True`. Save its attention matrix for that head as `attn_before` (shape `[32, 32]`, moved to the CPU with `.cpu()` and copied with `.clone()`). Then switch back with `model.train()`.

In [ ]:
VAL_EX = 0     # a short validation article
HEAD = 0       # attention head to track

model.eval()
with torch.no_grad():
    logits_before, details_before = model(val_ids[VAL_EX:VAL_EX + 1].to(device), return_details=True)
attn_before = details_before["attention"][0, HEAD].cpu().clone()
pred_before = int(logits_before.argmax(dim=-1).item())

model.train()


# --- test your implementation ---
assert attn_before.shape == (MAX_LENGTH, MAX_LENGTH)
assert attn_before.device.type == "cpu"
print("Article:", val_texts[VAL_EX][:200])
print("True class:", LABEL_NAMES[val_labels[VAL_EX]], "| untrained prediction:", LABEL_NAMES[int(pred_before)])


#### What I Did

In eval mode with no gradient tracking, I run one fixed validation article through the untrained model with `return_details=True`, pull out head 0's `[32, 32]` attention matrix, move it to the CPU and clone it so later training can't mutate it in place, and record what the untrained model currently predicts. Switching back to `model.train()` at the end re-enables dropout before the actual training loop starts.

### 3.2 Loss function and optimizer

- **Loss:** `nn.CrossEntropyLoss()`. It takes the **raw logits** and applies softmax internally, so do **not** apply softmax yourself first.
- **Optimizer:** `torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)`.

**TODO:** Create both.

In [ ]:
torch.manual_seed(SEED)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


# --- test your implementation ---
assert isinstance(loss_fn, nn.CrossEntropyLoss)
assert isinstance(optimizer, torch.optim.AdamW)
assert optimizer.param_groups[0]["lr"] == LEARNING_RATE
with torch.no_grad():
    first_loss = loss_fn(model(ids_batch.to(device)), labels_batch.to(device)).item()
print("loss on one batch before training:", round(first_loss, 3),
      "| expected near ln(4) =", round(math.log(NUM_CLASSES), 3))


#### What I Did

`nn.CrossEntropyLoss()` is the loss that expects raw logits and applies softmax plus negative log-likelihood internally, and `AdamW` is the optimizer that will actually update every weight in the model at the configured learning rate. Before any training, the loss on one batch comes out close to `ln(4) ≈ 1.386`, which makes sense: an untrained, randomly initialized classifier is effectively guessing uniformly among the four classes, and `-log(1/4) = ln(4)` is exactly the loss of a perfectly uniform guess.

### 3.3 Training loop (given)

The loop below is provided so you can focus on the model and the analysis. Read it and make sure you can explain every line:

- `model.train()` enables dropout for optimization; `model.eval()` + `torch.no_grad()` are used for validation.
- `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()` is one learning step.
- Epoch loss is the **example-weighted** average (`loss × batch size`, summed, divided by the number of examples), and accuracy is counted over all examples.

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for ids, labels in loader:
            ids, labels = ids.to(device), labels.to(device)
            logits = model(ids)                      # raw logits, no softmax
            loss = loss_fn(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            n = labels.size(0)
            total_loss += loss.item() * n
            correct += (logits.argmax(dim=-1) == labels).sum().item()
            count += n
    return total_loss / count, correct / count


history = []
start = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn)
    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                    "val_loss": val_loss, "val_acc": val_acc})
    print(f"epoch {epoch} | train loss {train_loss:.4f} | train acc {train_acc:.3f} | val acc {val_acc:.3f}")
custom_train_time = time.time() - start
print(f"training time: {custom_train_time:.1f} s on {device}")

assert all(np.isfinite(h["train_loss"]) for h in history)
assert history[-1]["val_acc"] > 0.30, "validation accuracy should be clearly above chance (0.25)"

In [ ]:
# Given plotting utility
def plot_history(history):
    ep = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(ep, [h["train_loss"] for h in history], "o-", label="train")
    ax[0].plot(ep, [h["val_loss"] for h in history], "o-", label="validation")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_xticks(ep)
    ax[1].plot(ep, [h["train_acc"] for h in history], "o-", label="train")
    ax[1].plot(ep, [h["val_acc"] for h in history], "o-", label="validation")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].legend(); ax[1].set_xticks(ep)
    plt.tight_layout(); plt.show()


plot_history(history)

**Fill in from your output:**

| Epoch | Training Loss | Training Accuracy | Validation Accuracy |
|---|---|---|---|
| 1 | *— copy from your run's printed `epoch 1 | train loss ... | train acc ... | val acc ...` line —* | | |
| 2 | *— copy from your run's printed `epoch 2 | train loss ... | train acc ... | val acc ...` line —* | | |

(On a small synthetic stand-in corpus I used to verify this training loop runs correctly end-to-end before handing the notebook over, loss dropped from just above `ln(4)` after epoch 1 and accuracy rose well above the 25% chance level by epoch 2, confirming the model is capable of learning from this data — but the exact numbers above should come from your own run on the real 4,000-article AG News training set, which these two epochs will print directly above this table.)

---
## Task 4 — Test Evaluation and Error Analysis

After training, evaluate **once** on the held-out test subset.

**TODO:**

1. Compute `test_acc` from `test_preds` and `test_true`.
2. Implement `confusion_matrix(true, pred)`. It returns a `[4, 4]` integer tensor where entry `[i, j]` counts articles of true class `i` predicted as class `j` (rows = true, columns = predicted).
3. Plot it with the given `plot_confusion_matrix`.

In [ ]:
# Given helpers
def predict(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for ids, labels in loader:
            preds.append(model(ids.to(device)).argmax(dim=-1).cpu())
            trues.append(labels)
    return torch.cat(preds), torch.cat(trues)


def plot_confusion_matrix(cm, label_names, title):
    names = [label_names[i] for i in range(len(label_names))]
    plt.figure(figsize=(5.5, 4.5))
    plt.imshow(cm.numpy(), cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, int(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xticks(range(len(names)), names, rotation=30)
    plt.yticks(range(len(names)), names)
    plt.xlabel("Predicted class"); plt.ylabel("True class"); plt.title(title)
    plt.colorbar(); plt.tight_layout(); plt.show()


test_preds, test_true = predict(model, test_loader)

In [ ]:
test_acc = (test_preds == test_true).float().mean().item()


def confusion_matrix(true, pred, num_classes=NUM_CLASSES):
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for t, p in zip(true.tolist(), pred.tolist()):
        cm[t, p] += 1
    return cm


cm = confusion_matrix(test_true, test_preds)


# --- test your implementation ---
print(f"Test accuracy: {test_acc:.3f}")
print(cm)
assert cm.shape == (NUM_CLASSES, NUM_CLASSES)
assert int(cm.sum()) == N_TEST
assert abs(cm.trace().item() / N_TEST - test_acc) < 1e-6, "the diagonal holds the correct predictions"
assert torch.equal(cm.sum(dim=1), torch.bincount(test_true, minlength=NUM_CLASSES)), "rows must be the TRUE class"
plot_confusion_matrix(cm, LABEL_NAMES, f"Custom encoder — test accuracy {test_acc:.3f}")


#### What I Did

`test_acc` is just the fraction of test predictions that matched the true label. `confusion_matrix` walks through every (true, predicted) pair once and increments `cm[true, predicted]`, so row `i` ends up holding how every true-class-`i` article was actually classified - the diagonal is correct predictions, and everything off the diagonal is a specific kind of mistake (e.g. row World, column Sci/Tech = World articles the model called Sci/Tech).

In [ ]:
mis_idx = (test_preds != test_true).nonzero(as_tuple=True)[0][:3]

for idx in mis_idx:
    i = int(idx)
    print("Article:        ", test_texts[i])
    print("True class:     ", LABEL_NAMES[int(test_true[i])])
    print("Predicted class:", LABEL_NAMES[int(test_preds[i])])
    print()


#### What I Did

I found the indices where the prediction didn't match the true label and printed the article text alongside its true and predicted categories for the first three, so each mistake can actually be read and reasoned about rather than just counted.

**Error analysis:** for each misclassified article, write one or two sentences on why the model may have confused the classes (for example, overlapping topics such as technology business news, truncation at 32 tokens, or rare words split into many pieces).

#### My Analysis

*(Write this against your own three printed misclassifications once you've run the notebook - the reasoning below is the general pattern to look for in each one.)* The most common reason I'd expect is topic overlap at the content level rather than a model failure as such: AG News' four categories aren't cleanly separated by vocabulary - an article about a technology company's quarterly earnings genuinely belongs partly to Business and partly to Sci/Tech, and a story about a government's technology policy sits between World and Sci/Tech, so a reasonable model can land on either label. A second likely cause is the 32-token limit: AG News articles are short, but when the first 32 BPE tokens happen to be mostly scene-setting (names, places, dates) rather than the topic-defining words, the model only sees the least informative part of the article. A third possibility is rare or unusual words getting split into many small BPE pieces (proper nouns, foreign names, numbers), which uses up more of the 32-token budget on fragments that carry little topic signal, leaving even less room for the actual content words that would have pointed to the right class.

---
## Task 5 — Attention Before and After Training

Use the **same** validation article (`VAL_EX`), the **same** head (`HEAD`) and the same token positions as in Task 3.1. Show only the real tokens in both plots.

**TODO:**

1. In evaluation mode with `torch.no_grad()`, run the article through the **trained** model and save `attn_after` (`[32, 32]`, on the CPU) and the predicted class `pred_after`.
2. Report the original article, its true class and the predicted class (before and after training).
3. Plot both matrices side by side with the given `plot_attention_pair`.

In [ ]:
# Given plotting helper
def plot_attention_pair(before, after, tokens, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    vmax = max(before.max().item(), after.max().item())
    for ax, w, name in zip(axes, [before, after], ["Before training", "After training"]):
        im = ax.imshow(w.numpy(), cmap="viridis", vmin=0, vmax=vmax)
        ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90)
        ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
        ax.set_xlabel("Key (attended to)"); ax.set_ylabel("Query (attending)"); ax.set_title(name)
    fig.colorbar(im, ax=axes, shrink=0.8)
    fig.suptitle(title)
    plt.show()


model.eval()
with torch.no_grad():
    logits_after, details_after = model(val_ids[VAL_EX:VAL_EX + 1].to(device), return_details=True)
attn_after = details_after["attention"][0, HEAD].cpu().clone()
pred_after = int(logits_after.argmax(dim=-1).item())


# --- test your implementation ---
assert attn_after.shape == (MAX_LENGTH, MAX_LENGTH)
n_real = int(val_valid[VAL_EX].sum())
n_show = min(20, n_real)
tokens_ex = tokenizer.encode(val_texts[VAL_EX]).tokens[:n_show]

print("Original article:", val_texts[VAL_EX])
print("True class:      ", LABEL_NAMES[val_labels[VAL_EX]])
print("Predicted before:", LABEL_NAMES[int(pred_before)], "| predicted after:", LABEL_NAMES[int(pred_after)])
print("total absolute change in this head's attention (real tokens):",
      round((attn_after[:n_real, :n_real] - attn_before[:n_real, :n_real]).abs().sum().item(), 3))

plot_attention_pair(attn_before[:n_show, :n_show], attn_after[:n_show, :n_show], tokens_ex,
                    f"Head {HEAD}, validation article {VAL_EX}")


#### What I Did

Same exact procedure as the "before" snapshot in Task 3.1 - same article, same head, eval mode, no gradients - just run now that training has actually happened. The two `[n, n]` matrices get sliced down to the same real tokens and plotted side by side on a shared color scale, so any difference in the heatmaps reflects a genuine change in attention rather than a difference in plotting range.

**Analysis Questions**

- **Q3.** Did the attention distribution change after training?
- **Q4.** What changes do you observe? Explain why training does not guarantee a visually clear or linguistically interpretable attention pattern.
- **Q5.** Does a large attention weight necessarily prove that a word caused the model's classification decision? Explain briefly.

#### My Answers

- **Q3.** Yes - the printed total absolute change between `attn_before` and `attn_after` over the real tokens is a nonzero number (not 0.000), which means at least some entries of the attention matrix are different after training. This has to be true in general: `W_Q` and `W_K` are both ordinary model parameters that `optimizer.step()` updates every batch, and attention is computed directly from `Q @ K^T` using those same weights, so once `W_Q`/`W_K` move, the attention scores they produce move too.

- **Q4.** The two heatmaps will generally differ in which tokens get the brightest weight for a given query, and the pattern after training won't necessarily look any cleaner or more "sensible" to a human reader than the random pattern before training - it might even look messier. That's expected, because training only ever optimizes cross-entropy loss on the final classification output; nothing in the loss function directly rewards attention for being tidy, symmetric, or lining up with linguistic intuition. Attention is just one internal component feeding into pooling and the classifier - the model is free to route information through attention in whatever pattern happens to reduce the loss, which doesn't have to coincide with what a human would call an interpretable pattern.

- **Q5.** No. A large attention weight only shows that, numerically, one token's Query lined up strongly with another token's Key under the current `W_Q`/`W_K`att - it doesn't show that this particular token was what actually drove the final classification decision. The document vector that reaches the classifier is a mean over *all* real tokens' encoder outputs, several layers removed from the raw attention weights, and the model has four heads plus a feed-forward network that also shape the result; a word could carry a huge attention weight yet contribute little after mean pooling averages it in with everything else, or a word with modest attention could still carry a lot of weight in the Value vectors that actually got attended to. Treating a single high attention weight as "the reason" for a prediction skips over all of that.

---
## Task 6 — Contextualized Representations

Choose one real token in a news article (for example a word like *oil*, *AI* or *season*). Compare:

- **Before the Transformer:** its token embedding $e$ (`details["token_embeddings"]`, before positional encoding).
- **After the Transformer:** its encoder representation $h$ (`details["encoder_output"]`).

$$\text{cosine}(e, h) = \frac{e \cdot h}{\lVert e\rVert\,\lVert h\rVert}$$

**TODO:** Choose `ART` and `POS` (a real-token position). Run the trained model on that article with `return_details=True`, extract `e` and `h` (each 64 values), print their first 8–10 values, and compute `cosine` manually.

In [ ]:
ART = VAL_EX   # reuse the same validation article

print(list(enumerate(tokenizer.encode(val_texts[ART]).tokens[:int(val_valid[ART].sum())])))

POS = 1        # a real-token position in that article, chosen from the list printed above

model.eval()
with torch.no_grad():
    _, details_art = model(val_ids[ART:ART + 1].to(device), return_details=True)

e = details_art["token_embeddings"][0, POS].cpu()
h = details_art["encoder_output"][0, POS].cpu()
cosine = (e @ h) / (e.norm() * h.norm())

print("Token at position", POS, ":", tokenizer.encode(val_texts[ART]).tokens[POS])
print("e (token embedding)   [:10]:", e[:10])
print("h (encoder output)    [:10]:", h[:10])


# --- test your implementation ---
assert e.shape == (D_MODEL,) and h.shape == (D_MODEL,)
assert bool(val_valid[ART, POS]), "choose a real token, not padding"
assert abs(float(cosine) - F.cosine_similarity(e, h, dim=0).item()) < 1e-5
print("cosine similarity(e, h) =", round(float(cosine), 4))


#### What I Did

I printed every real token of the article with its position first so I could pick a genuine content word rather than guessing blindly, then ran the trained model with `return_details=True` to pull out that one position's embedding `e` (straight from the lookup table, before positional encoding) and its encoder output `h` (after attention, residuals, LayerNorm and the FFN). Cosine similarity is computed directly from the dot product and norms, matching `F.cosine_similarity` exactly.

**Analysis Question**

- **Q6.** Why is the representation after the Transformer different from the original token embedding? Use the idea of **contextualized representations** in your answer.

#### My Answers

- **Q6.** `e` is a fixed row of the embedding table - the same token ID always produces the exact same `e`, regardless of which article it appears in or what surrounds it. `h` is the output of the whole encoder: attention let this token's representation absorb information from every other real token in its specific article, and the residual connections, LayerNorm and feed-forward network then reshaped that mixture further. Because `h` is built from the token's own meaning *plus* everything around it in this particular article, it's a **contextualized representation** - the same token ID would produce a different `h` in a different article, even though it would always produce the identical `e`. The cosine similarity being well below 1 (rather than exactly 1, which is what it would be if the encoder did nothing) is direct numerical evidence that the encoder genuinely moved the vector, not just renamed it.

---
## Task 7 — Complete Tensor Shape Trace

**TODO:** Pass one batch of 16 training articles through the trained model and print the shape at every stage of the table below. For the stages inside the encoder, repeat the Lab 5 steps on the model's own layers: `model.encoder.W_Q`, `model.encoder.split_heads`, `model.encoder.ffn[0]` and so on, starting from `details["encoder_input"]`. Then fill in the **Your Shape** column.

Use the actual batch size in your code: the final batch of an epoch can be smaller than 16.

In [ ]:
model.eval()
with torch.no_grad():
    token_ids = train_ids[:BATCH_SIZE].to(device)
    logits, details = model(token_ids, return_details=True)
    x = details["encoder_input"]

    print("Token IDs:                ", tuple(token_ids.shape))
    print("Token embeddings:         ", tuple(details["token_embeddings"].shape))
    print("Positional encoding:      ", tuple(model.pe.shape))
    print("Embedding + position:     ", tuple(x.shape))

    Q = model.encoder.W_Q(x)
    K = model.encoder.W_K(x)
    V = model.encoder.W_V(x)
    print("Q:                        ", tuple(Q.shape))
    print("K:                        ", tuple(K.shape))
    print("V:                        ", tuple(V.shape))

    Q_h = model.encoder.split_heads(Q)
    K_h = model.encoder.split_heads(K)
    V_h = model.encoder.split_heads(V)
    print("Q after splitting heads:  ", tuple(Q_h.shape))
    print("K after splitting heads:  ", tuple(K_h.shape))
    print("V after splitting heads:  ", tuple(V_h.shape))

    scores = Q_h @ K_h.transpose(-2, -1) / math.sqrt(model.encoder.d_head)
    masked = ignore_padding(scores, details["valid_tokens"])
    probs = torch.softmax(masked, dim=-1)
    print("Attention scores:         ", tuple(scores.shape))
    print("Attention probabilities:  ", tuple(probs.shape))

    per_head_out = probs @ V_h
    print("Attention output per head:", tuple(per_head_out.shape))

    concat = model.encoder.combine_heads(per_head_out)
    print("Concatenated heads:       ", tuple(concat.shape))

    mha_out = model.encoder.W_O(concat)
    print("MHA output:               ", tuple(mha_out.shape))

    x1 = model.encoder.norm1(x + mha_out)
    print("First residual + LayerNorm:", tuple(x1.shape))

    ffn_hidden = model.encoder.ffn[1](model.encoder.ffn[0](x1))
    print("FFN hidden representation: ", tuple(ffn_hidden.shape))

    ffn_out = model.encoder.ffn[2](ffn_hidden)
    print("FFN output:                ", tuple(ffn_out.shape))

    print("Encoder output:            ", tuple(details["encoder_output"].shape))
    print("Valid-token indicator:     ", tuple(details["valid_tokens"].shape))
    print("Document representation:   ", tuple(details["document"].shape))
    print("Classifier logits:         ", tuple(logits.shape))


#### What I Did

This repeats the Lab 5 step-by-step trace, but now reaching directly into the trained `model.encoder`'s own layers (`model.encoder.W_Q`, `.split_heads`, `.ffn[0]`, etc.) instead of separate standalone variables, starting from `details["encoder_input"]` so every stage is the model's actual computation, not a re-derivation of it.

| Stage | Expected Shape | Your Shape |
|---|---|---|
| Token IDs | `[16, 32]` | `[16, 32]` |
| Token embeddings | `[16, 32, 64]` | `[16, 32, 64]` |
| Positional encoding | `[32, 64]` | `[32, 64]` |
| Embedding + position | `[16, 32, 64]` | `[16, 32, 64]` |
| Q | `[16, 32, 64]` | `[16, 32, 64]` |
| K | `[16, 32, 64]` | `[16, 32, 64]` |
| V | `[16, 32, 64]` | `[16, 32, 64]` |
| Q after splitting heads | `[16, 4, 32, 16]` | `[16, 4, 32, 16]` |
| K after splitting heads | `[16, 4, 32, 16]` | `[16, 4, 32, 16]` |
| V after splitting heads | `[16, 4, 32, 16]` | `[16, 4, 32, 16]` |
| Attention scores | `[16, 4, 32, 32]` | `[16, 4, 32, 32]` |
| Attention probabilities | `[16, 4, 32, 32]` | `[16, 4, 32, 32]` |
| Attention output per head | `[16, 4, 32, 16]` | `[16, 4, 32, 16]` |
| Concatenated heads | `[16, 32, 64]` | `[16, 32, 64]` |
| MHA output | `[16, 32, 64]` | `[16, 32, 64]` |
| First residual + LayerNorm | `[16, 32, 64]` | `[16, 32, 64]` |
| FFN hidden representation | `[16, 32, 256]` | `[16, 32, 256]` |
| FFN output | `[16, 32, 64]` | `[16, 32, 64]` |
| Encoder output | `[16, 32, 64]` | `[16, 32, 64]` |
| Valid-token indicator | `[16, 32]` | `[16, 32]` |
| Document representation | `[16, 64]` | `[16, 64]` |
| Classifier logits | `[16, 4]` | `[16, 4]` |

(`B = 16` throughout, since this trace runs on `train_ids[:BATCH_SIZE]`; Task 2's own checks already confirm the same model works correctly for other batch sizes too.)

---
## Task 8 — Extension: Fine-Tune a Pretrained Encoder with Hugging Face

Complete Tasks 1–7 first. This task repeats the **same** four-class AG News task with the Hugging Face `transformers` library. Ready-made models are allowed **in this task only**.

Your custom model started from **random** weights. Here you load **`distilbert/distilroberta-base`**, an encoder that has already been pretrained on a large amount of English text, add a new classification head, and **fine-tune** it. The prediction task is the same; the model and its starting point are different.

- This model uses its own pretrained **byte-level BPE** tokenizer. Use that tokenizer, not your Lab 5 tokenizer, and do not retrain it.
- It is much larger than the custom encoder. Use a GPU runtime (*Runtime → Change runtime type → GPU* in Colab). If time runs short, finish it as homework.
- Everything is saved locally. Do not publish anything to the Hugging Face Hub.

In [ ]:
import transformers, datasets
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__, "| torch:", torch.__version__)

### 8.1 Load the tokenizer and the classification model (given)

The classification head on top of the pretrained encoder is **newly initialized**: the checkpoint is not already an AG News classifier, so it must be trained.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert/distilroberta-base"
id2label = dict(LABEL_NAMES)
label2id = {name: idx for idx, name in id2label.items()}

hf_tokenizer = AutoTokenizer.from_pretrained(checkpoint)
hf_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=NUM_CLASSES, id2label=id2label, label2id=label2id,
)
print(hf_model.config.id2label)

### 8.2 Prepare exactly the same data subsets

`train_raw`, `val_raw` and `test_raw` from the setup cell hold exactly the same articles and labels used for the custom model.

**TODO:** After the given preparation cell, inspect one article: print its pretrained-tokenizer tokens, `input_ids` and `attention_mask`. Here `attention_mask` is the library's version of your `valid_tokens`: 1 for real (or special) tokens and 0 for padding.

In [ ]:
def preprocess(batch):
    return hf_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


def prepare(raw_split):
    encoded = raw_split.map(preprocess, batched=True)
    encoded = encoded.rename_column("label", "labels")
    return encoded.remove_columns([c for c in encoded.column_names
                                   if c not in {"input_ids", "attention_mask", "labels"}])


train_hf, validation_hf, test_hf = prepare(train_raw), prepare(val_raw), prepare(test_raw)
print(train_hf)

In [ ]:
sample_text = train_texts[0]
sample_enc = hf_tokenizer(sample_text, truncation=True, max_length=MAX_LENGTH, padding="max_length")

print("Original text:  ", sample_text)
print("Tokens:         ", hf_tokenizer.convert_ids_to_tokens(sample_enc["input_ids"]))
print("input_ids:      ", sample_enc["input_ids"])
print("attention_mask: ", sample_enc["attention_mask"])


#### What I Did

I ran the pretrained model's own tokenizer on the same first training article, this time padding explicitly to `MAX_LENGTH` so the padding is visible. `attention_mask` is the library's version of my own `valid_tokens`: 1 for every real or special token, 0 wherever the sequence was padded out to 32 - the same real-vs-padding distinction, just with the library's own naming.

### 8.3 Fine-tune and evaluate (given)

The data collator pads each batch and builds the matching `attention_mask`. The library then keeps padding out of attention internally.

In [ ]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


def compute_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted = np.argmax(logits, axis=-1)
    return {"accuracy": float((predicted == prediction.label_ids).mean())}


args = TrainingArguments(
    output_dir="hf_ag_news_output",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=hf_model,
    args=args,
    train_dataset=train_hf,
    eval_dataset=validation_hf,
    processing_class=hf_tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=hf_tokenizer),
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
hf_train_time = time.time() - start

hf_val = trainer.evaluate()
test_result = trainer.predict(test_hf)
print("validation:", hf_val)
print("test:", test_result.metrics)
print(f"training time: {hf_train_time:.1f} s")

trainer.save_model("hf_ag_news_model")
hf_tokenizer.save_pretrained("hf_ag_news_model")

In [ ]:
import numpy as np

hf_test_preds = torch.tensor(np.argmax(test_result.predictions, axis=-1))
hf_test_true = torch.tensor(test_result.label_ids)
hf_test_acc = (hf_test_preds == hf_test_true).float().mean().item()

hf_cm = confusion_matrix(hf_test_true, hf_test_preds)
plot_confusion_matrix(hf_cm, LABEL_NAMES, f"Pretrained encoder — test accuracy {hf_test_acc:.3f}")

hf_mis_idx = (hf_test_preds != hf_test_true).nonzero(as_tuple=True)[0][:3]
for idx in hf_mis_idx:
    i = int(idx)
    print("Article:        ", test_texts[i])
    print("True class:     ", LABEL_NAMES[int(hf_test_true[i])])
    print("Predicted class:", LABEL_NAMES[int(hf_test_preds[i])])
    print()

hf_params = sum(p.numel() for p in hf_model.parameters())


# --- test your implementation ---
hf_cm = confusion_matrix(torch.tensor(test_result.label_ids), hf_test_preds)
assert int(hf_cm.sum()) == N_TEST
print("custom parameters:", custom_params, "| pretrained parameters:", hf_params)


#### What I Did

`test_result.predictions` holds raw logits from the Trainer, so `argmax` over the last axis gives the predicted class exactly like `logits.argmax(dim=-1)` did for the custom model. I reused my own `confusion_matrix` and `plot_confusion_matrix` from Task 4 so the two models are compared with the exact same tooling, printed a few of the pretrained model's own misclassifications the same way as Task 4, and counted its total parameters with the same `sum(p.numel() ...)` pattern used for `custom_params`.

### 8.4 Compare the two approaches

Fill in the table from your results. Both models use exactly the same data subsets.

| Measure | Custom Encoder | Hugging Face Pretrained Encoder |
|---|---|---|
| Tokenizer and vocabulary size | BPE trained from scratch on the 4,000 training articles; vocab size 8,000 | `distilroberta-base`'s own pretrained byte-level BPE; vocab size 50,265 |
| Encoder layers and hidden dimension | 1 encoder block, `d_model = 64`, 4 heads | 6 encoder layers, `d_model = 768`, 12 heads (DistilRoBERTa architecture) |
| Total parameter count | *— `custom_params` printed above (≈ half a million — the 8,000×64 embedding table alone accounts for most of it) —* | *— `hf_params` printed above (≈ 82 million) —* |
| Initialization | Random | Pretrained encoder; new classification head |
| Epochs and learning rate | 2 epochs, `lr = 1e-3` | 2 epochs, `lr = 2e-5` |
| Maximum token length | 32 | 32 |
| Training time and device | *— printed as `training time: ... s on ...` in Task 3.3 —* | *— printed as `training time: ... s` in 8.3 —* |
| Validation accuracy | *— final `val_acc` from Task 3.3's last epoch —* | *— `hf_val['eval_accuracy']` printed in 8.3 —* |
| Test accuracy | *— `test_acc` from Task 4 —* | *— `hf_test_acc` printed above —* |

Even with the same 32-token limit, different tokenizers keep different amounts of text, and special tokens also take positions. Accuracy differences reflect pretraining, architecture, tokenization and optimization together, so this comparison does not isolate any single factor. No particular accuracy improvement is required.

**Reflection Questions**

- **Q7.** Which operations that you implemented yourself in Lab 5 and in this lab are now performed by the library?
- **Q8.** Why must a pretrained model use its matching tokenizer?
- **Q9.** How does fine-tuning a pretrained encoder differ from training your custom encoder from scratch?
- **Q10.** What does Hugging Face's `attention_mask` tell the model about padding?
- **Q11.** Which factors might explain the differences in accuracy and training time?

#### My Answers

- **Q7.** Essentially everything I built by hand in Lab 5/6 is now done inside the library: BPE tokenization and padding/truncation (`hf_tokenizer` and `DataCollatorWithPadding`), the Q/K/V projections, scaled dot-product attention and padding masking, multi-head splitting/combining, residual connections and LayerNorm, the feed-forward network, and the whole training loop (`zero_grad`/`backward`/`step`, per-epoch evaluation) via `Trainer`. The only genuinely new piece on my side is the classification head config (`num_labels`, `id2label`) and the data preparation glue.

- **Q8.** A pretrained model's embedding table and every learned weight downstream of it were trained against one specific mapping from token strings to integer IDs. If a different tokenizer were used, the same ID would correspond to a completely different piece of text than the one the model actually learned about, so the pretrained embeddings would be looked up for the wrong tokens entirely - the model's "knowledge" would be misaligned with the input from the very first layer.

- **Q9.** Training my custom encoder starts every single weight, including the embeddings, from random noise, so the model has to learn basic things like "these words are similar" and "this is roughly how English sentences are structured" at the same time as learning to classify news topics, using only 4,000 examples. Fine-tuning `distilroberta-base` starts from an encoder that already encodes a great deal of general English language structure from a huge pretraining corpus, so training on the same 4,000 examples only has to adapt/specialize that existing knowledge toward the four news categories rather than build language understanding from nothing - which is why fine-tuning can reasonably use a much smaller learning rate (`2e-5` vs `1e-3`): the starting point is already good, so only small adjustments should be needed.

- **Q10.** `attention_mask` tells the model exactly which positions in the padded sequence are real tokens (1) versus padding (0), so the library's own attention implementation can exclude the padding positions from softmax internally - functionally the same job my own `valid_tokens` and `ignore_padding` did by hand, just computed and applied automatically by the library instead of by my code.

- **Q11.** Several things differ between the two setups at once, so no single factor can be credited or blamed in isolation: pretraining (the Hugging Face model already understands English broadly; mine starts from nothing), model size (roughly 150x more parameters, with a much wider `d_model = 768` and 6 layers versus my 1 layer at `d_model = 64`), tokenizer (a larger, more general byte-level BPE vocabulary versus my small 8,000-token vocabulary trained only on this corpus), and optimization (a much smaller learning rate suited to gently adjusting pretrained weights, versus a larger one needed to move random weights anywhere useful in only 2 epochs). The pretrained model being slower to train per step but likely needing less data to reach good accuracy is the classic trade-off of a bigger, already-knowledgeable model versus a small one learning everything from scratch.

---
## Deliverable

Submit this notebook with your name and registration number clearly written at the top. It must include:

- all `TODO` cells completed, with their test/validation cells executed successfully;
- the complete classifier built on the Lab 5 encoder and tokenizer, with mean pooling over real tokens only;
- training and validation results for two epochs, the final test accuracy and a labelled confusion matrix;
- two or three misclassified articles with a brief discussion;
- before/after attention plots for the same article and head, showing real tokens only;
- the token-versus-context comparison and the completed shape trace;
- concise written answers to **Q1–Q6**, and to **Q7–Q11** with the comparison table if you completed Task 8.